# Lab Assignment 12
## Mixture of Experts inside a Transformer: routing, load balancing and capacity

---

### Objective

The tutorial built a mixture of experts on a toy: four clusters, four linear experts, and a router that
learned to send each cluster to its own expert. It ended on a failure — with an unlucky start, the
router overloaded one expert and the accuracy dropped.

This assignment moves MoE to where it is used in real language models: the **feed-forward layer of a
Transformer block**, with **every token routed separately**, exactly like the lecture's
*"Sparsely activated MoE Transformer block"* slide. The task is classifying news articles into four
topics.

You will build the layer yourself and then test four claims from the lecture:

1. Noisy top-k gating trains the router — **for every k?** (Task 2)
2. The **load-balancing loss** from the Switch Transformer spreads tokens evenly over the experts. (Task 3)
3. Uneven routing makes tokens **overflow an expert's capacity** and get dropped. (Task 4)
4. Experts **do not specialise by topic** (the Mixtral slides). (Task 5)

### How this differs from the tutorial

| | Tutorial demo | This assignment |
|---|---|---|
| Data | 2-D toy points | **AG News** articles: World, Sports, Business, Sci/Tech |
| Where the MoE sits | it is the whole model | the **feed-forward layer of a Transformer block** |
| What is routed | one point | **every token**, separately |
| Experts | 4 straight lines | **8** feed-forward networks |
| Router | softmax, top-1 | **noisy top-k** (Shazeer et al.), then the **Switch** router |
| Computation | all experts run, then 3 zeroed | only the chosen experts run (**sparse dispatch**) |
| Measured | accuracy, cluster table | router learning, **load balance**, **dropped tokens**, specialisation |

### Marks

| Task | Description | Marks |
|---|---|---|
| **Task 1** | Build the MoE layer (noisy top-k gating, sparse dispatch) | **4** |
| **Task 2** | Train it — and find the top-1 trap | **4** |
| **Task 3** | The load-balancing loss | **4** |
| **Task 4** | Expert capacity and dropped tokens | **4** |
| **Task 5** | What did the experts learn? | **4** |
| | **Total** | **20** |

---

### Before you start

- **You need the file `mini_agnews.tsv`** in the same folder as this notebook: 16,000 training and
  2,000 test articles, 4,000 and 500 per topic. **In Colab:** click the folder icon in the left sidebar
  and drag the file in, or run
  ```python
  from google.colab import files
  files.upload()
  ```
  Use this exact file. Do not download AG News yourself — the numbers you compare depend on this
  sample, and it has been cleaned so that no test article also appears in the training set.
- A GPU is **not** required. Each model trains in roughly 10–40 seconds on a laptop or Colab CPU, and
  you train six of them.
- The **Setup** cell gives you the data pipeline, the Transformer block and the training loop, so
  your time goes on the MoE parts. Read it once: your layer must plug into it.
- Everything is seeded, so re-running gives the same numbers. Your numbers may still differ slightly
  from a classmate's if you write the layer differently — the **patterns** are what is marked.

**You are expected to write the code yourself.** Each task states what to build, the names to use,
and the exact output it must print — but not how to write it.

---
## Setup

Run this once. It is given to you — do not change it.

In [1]:
import re, time, collections
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F

TOPICS    = ['World', 'Sports', 'Business', 'Sci/Tech']
MAX_LEN   = 64      # tokens kept per article
D_MODEL   = 64      # size of every token vector
D_HIDDEN  = 128     # hidden size inside each expert
N_EXPERTS = 8
EPOCHS    = 3
BATCH     = 64      # articles per batch

# ---------------- data ----------------
def tokenize(text):
    return re.findall(r"[a-z0-9]+(?:'[a-z]+)?|[^\sa-z0-9]", text.lower())

rows = [line.rstrip('\n').split('\t') for line in open('mini_agnews.tsv', encoding='utf-8')][1:]
train_rows = [(tokenize(t), TOPICS.index(l)) for s, l, t in rows if s == 'train']
test_rows  = [(tokenize(t), TOPICS.index(l)) for s, l, t in rows if s == 'test']

counts = collections.Counter(w for toks, _ in train_rows for w in toks)
vocab = ['<pad>', '<unk>'] + [w for w, n in counts.most_common() if n >= 2]
word_index = {w: i for i, w in enumerate(vocab)}

def encode(data):
    X = torch.zeros(len(data), MAX_LEN, dtype=torch.long)          # 0 = <pad>, at the END
    for i, (toks, _) in enumerate(data):
        ids = [word_index.get(w, 1) for w in toks[:MAX_LEN]]       # 1 = <unk>
        X[i, :len(ids)] = torch.tensor(ids)
    return X, torch.tensor([label for _, label in data])

X_train, y_train = encode(train_rows)
X_test,  y_test  = encode(test_rows)
print(f"vocabulary {len(vocab)}   X_train {tuple(X_train.shape)}   X_test {tuple(X_test.shape)}")
print(f"real tokens per article: mean {(X_train != 0).sum(1).float().mean():.1f}")

# ---------------- model ----------------
class Expert(nn.Module):
    # One expert = the ordinary Transformer feed-forward network: Linear -> ReLU -> Linear.
    def __init__(self, d_model, d_hidden):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_hidden), nn.ReLU(), nn.Linear(d_hidden, d_model))
    def forward(self, x):
        return self.net(x)

class TinyTransformerClassifier(nn.Module):
    # Embedding -> one Transformer block (self-attention, then `ffn`) -> mean over tokens -> 4 topics.
    # `ffn` is either ONE Expert (the dense model) or your MoELayer. It receives only the REAL
    # tokens of the batch, flattened row by row into shape (T, D_MODEL).
    def __init__(self, vocab_size, ffn):
        super().__init__()
        self.emb  = nn.Embedding(vocab_size, D_MODEL, padding_idx=0)
        self.pos  = nn.Embedding(MAX_LEN, D_MODEL)
        self.attn = nn.MultiheadAttention(D_MODEL, num_heads=4, batch_first=True)
        self.ln1, self.ln2 = nn.LayerNorm(D_MODEL), nn.LayerNorm(D_MODEL)
        self.ffn  = ffn
        self.out  = nn.Linear(D_MODEL, len(TOPICS))

    def forward(self, X):
        mask = X != 0
        x = self.emb(X) + self.pos(torch.arange(X.shape[1]))
        a = self.ln1(x)
        x = x + self.attn(a, a, a, key_padding_mask=~mask, need_weights=False)[0]
        tokens = self.ln2(x)[mask]                 # (T, D_MODEL): real tokens only, row by row
        y = torch.zeros_like(x)
        y[mask] = self.ffn(tokens)
        x = x + y                                  # residual connection around the ffn
        return self.out((x * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True))

def make_model(make_ffn, seed=42):
    torch.manual_seed(seed)
    return TinyTransformerClassifier(len(vocab), make_ffn())

# ---------------- training and evaluation ----------------
def train_model(model, alpha=0.0, seed=42):
    # Adam + cross-entropy. If alpha > 0 it also adds alpha * load_balancing_loss(...),
    # a function YOU write in Task 3.
    torch.manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=2e-3)
    start = time.time()
    for epoch in range(EPOCHS):
        model.train()
        order = torch.randperm(len(X_train))
        for b in range(0, len(X_train), BATCH):
            idx = order[b:b + BATCH]
            loss = F.cross_entropy(model(X_train[idx]), y_train[idx])
            if alpha > 0:
                f = model.ffn
                loss = loss + alpha * load_balancing_loss(f.probs, f.top1, f.n_experts)
            opt.zero_grad(); loss.backward(); opt.step()
    print(f"  trained in {time.time() - start:.0f} s")
    return model

@torch.no_grad()
def evaluate(model):
    # Returns (test accuracy, routing). `routing` is a list with one tensor per test batch of
    # BATCH articles: the chosen experts of every real token, shape (tokens, k), best first.
    # None for a dense model.
    model.eval()
    correct, routing = 0, []
    for b in range(0, len(X_test), BATCH):
        logits = model(X_test[b:b + BATCH])
        correct += (logits.argmax(1) == y_test[b:b + BATCH]).sum().item()
        if hasattr(model.ffn, 'top_idx'):
            routing.append(model.ffn.top_idx.clone())
    return correct / len(X_test), (routing or None)

# ---------------- the dense baseline ----------------
print("\nDense baseline (one feed-forward network, no experts):")
dense = train_model(make_model(lambda: Expert(D_MODEL, D_HIDDEN)))
dense_acc, _ = evaluate(dense)
print(f"  test accuracy {dense_acc:.4f}")

vocabulary 19359   X_train (16000, 64)   X_test (2000, 64)
real tokens per article: mean 43.8

Dense baseline (one feed-forward network, no experts):


  trained in 5 s
  test accuracy 0.8505


---
# Task 1: Build the MoE layer &nbsp;&nbsp;*(4 marks)*

Two classes. The **router** decides, for each token, which experts to use and how much to trust each
one. The **MoE layer** sends each token to its chosen experts and adds up the weighted answers.

**The router — noisy top-k gating** (Shazeer et al. 2017, the *Noisy Top-K Gating* slides in part 1):

$$H(x)_i = (x\,W_g)_i + \text{StandardNormal}()\cdot\text{Softplus}\big((x\,W_{noise})_i\big)$$

$$\text{KeepTopK}(v, k)_i = \begin{cases} v_i & \text{if } v_i \text{ is among the top } k \\ -\infty & \text{otherwise}\end{cases}
\qquad\qquad G(x) = \text{Softmax}\big(\text{KeepTopK}(H(x), k)\big)$$

The noise is added **only while training**. The $-\infty$ makes the softmax give exactly 0 to every
expert that was not chosen, so each row of $G(x)$ has exactly $k$ non-zero entries that add up to 1.

**The MoE layer — sparse dispatch.** For each token $t$:

$$y_t = \sum_{i\ \in\ \text{chosen}(t)} G(x_t)_i \; E_i(x_t)$$

Unlike the demo, an expert must run **only on the tokens routed to it**. That is the whole point of
a sparse layer: with $k=2$ of 8 experts, each token costs 2 experts' worth of compute, not 8.

### What your code must create

| Name | Is |
|---|---|
| `NoisyTopKRouter(d_model, n_experts, k)` | has `w_gate` and `w_noise` (bias-free `nn.Linear(d_model, n_experts)`); `forward(x)` returns `gates, top_idx, probs` |
| `gates` | shape `(T, n_experts)`: exactly `k` non-zeros per row, each row sums to 1 |
| `top_idx` | shape `(T, k)`: the chosen experts, best first |
| `probs` | shape `(T, n_experts)`: softmax over **all** experts of the logits $H(x)$ — Task 3 needs it |
| `MoELayer(d_model, d_hidden, n_experts, router)` | has `.router`, `.experts` (an `nn.ModuleList` of `n_experts` `Expert`s) and `.n_experts` |
| `MoELayer.forward(x)` | returns `y` with the same shape as `x`, and stores `self.probs`, `self.top_idx` and `self.top1` (= `top_idx[:, 0]`, each token's first choice) |

### Output your code must print

Test the layer on `x = torch.randn(10, D_MODEL)` with `k = 2`, in training mode:

```
gates shape: (10, 8)
non-zeros per row: [2, 2, 2, 2, 2, 2, 2, 2, 2, 2]
row sums: all 1.0 -> True
output shape: (10, 64)

parameters in the MoE layer: 133,632
active per token (router + 2 experts): 34,176
dense feed-forward layer: 16,576
```

### Hints

- `logits.topk(k, dim=-1)` gives both the top values and their indices.
- `torch.full_like(logits, float('-inf'))` then `.scatter(-1, top_idx, top_values)` builds the
  KeepTopK vector in one line.
- `self.training` is `True` after `model.train()` and `False` after `model.eval()`.
- `(gates[:, i] > 0).nonzero().squeeze(1)` gives the row indices of the tokens routed to expert `i`.
- Count parameters with `sum(p.numel() for p in module.parameters())`.

In [2]:
# ---------------- TASK 1 ----------------

class NoisyTopKRouter(nn.Module):
    def __init__(self, d_model, n_experts, k):
        super().__init__()
        # bias-free gate and noise projections, d_model -> n_experts
        self.w_gate  = nn.Linear(d_model, n_experts, bias=False)
        self.w_noise = nn.Linear(d_model, n_experts, bias=False)
        self.k, self.n_experts = k, n_experts

    def forward(self, x):
        logits = self.w_gate(x)                                         # clean logits  x W_g
        if self.training:                                               # noise only while training
            logits = logits + torch.randn_like(logits) * F.softplus(self.w_noise(x))
        top_vals, top_idx = logits.topk(self.k, dim=-1)                 # best first
        kept = torch.full_like(logits, float('-inf')).scatter(-1, top_idx, top_vals)   # KeepTopK
        gates = F.softmax(kept, dim=-1)                                 # exactly k non-zeros, rows sum to 1
        probs = F.softmax(logits, dim=-1)                               # softmax over ALL experts (Task 3)
        return gates, top_idx, probs


class MoELayer(nn.Module):
    def __init__(self, d_model, d_hidden, n_experts, router):
        super().__init__()
        self.router = router
        self.n_experts = n_experts
        self.experts = nn.ModuleList([Expert(d_model, d_hidden) for _ in range(n_experts)])

    def forward(self, x):
        gates, top_idx, probs = self.router(x)
        self.probs, self.top_idx, self.top1 = probs, top_idx, top_idx[:, 0]
        y = torch.zeros_like(x)
        for i, expert in enumerate(self.experts):
            rows = (gates[:, i] > 0).nonzero().squeeze(1)               # tokens routed to expert i
            if rows.numel() == 0:
                continue
            out = expert(x[rows])                                       # sparse: ONLY those tokens
            y = y.index_add(0, rows, gates[rows, i].unsqueeze(1) * out)
        return y


# ---- checks on random input, k = 2, training mode ----
torch.manual_seed(0)
moe = MoELayer(D_MODEL, D_HIDDEN, N_EXPERTS, NoisyTopKRouter(D_MODEL, N_EXPERTS, k=2))
moe.train()
x = torch.randn(10, D_MODEL)
gates, top_idx, probs = moe.router(x)
y = moe(x)
print(f"gates shape: {tuple(gates.shape)}")
print(f"non-zeros per row: {(gates > 0).sum(1).tolist()}")
print(f"row sums: all 1.0 -> {torch.allclose(gates.sum(1), torch.ones(10))}")
print(f"output shape: {tuple(y.shape)}")

n_params = lambda m: sum(p.numel() for p in m.parameters())
n_moe    = n_params(moe)
n_active = n_params(moe.router) + 2 * n_params(moe.experts[0])
n_dense  = n_params(Expert(D_MODEL, D_HIDDEN))
print()
print(f"parameters in the MoE layer: {n_moe:,}")
print(f"active per token (router + 2 experts): {n_active:,}")
print(f"dense feed-forward layer: {n_dense:,}")


gates shape: (10, 8)
non-zeros per row: [2, 2, 2, 2, 2, 2, 2, 2, 2, 2]
row sums: all 1.0 -> True
output shape: (10, 64)

parameters in the MoE layer: 133,632
active per token (router + 2 experts): 34,176
dense feed-forward layer: 16,576


---
# Task 2: Train it — and find the top-1 trap &nbsp;&nbsp;*(4 marks)*

Now put the layer inside the Transformer and train it. The Switch Transformer slides make a big claim
for **top-1** routing: one expert per token is enough, and it is cheaper than two. So train the noisy
top-k router twice — once with `k = 2`, once with `k = 1` — and watch **whether the router learns
anything**.

To see that, copy the router's weights **before** training and measure how far they moved:

$$\text{router moved} = \lVert W_g^{\text{after}} - W_g^{\text{before}} \rVert$$

Then build a third router, the **Switch router** (Fedus et al. 2021), which routes to one expert in a
different way:

$$p(x) = \text{Softmax}(x\,W_g) \qquad i^* = \arg\max_i\, p(x)_i \qquad y = p(x)_{i^*}\,E_{i^*}(x)$$

No noise, and the gate value is the **softmax probability over all experts** — not a softmax over just
the one kept logit. This is the same top-1 rule as in the tutorial's section 3.

### What your code must create

| Name | Is |
|---|---|
| `SwitchRouter(d_model, n_experts)` | has `w_gate`; same `forward` interface as `NoisyTopKRouter` (with `k = 1`) |
| `noisy_k2`, `noisy_k1`, `switch_a0` | three trained `TinyTransformerClassifier`s with `MoELayer` ffns, all built with `make_model(..., seed=42)` and trained with `train_model(model)` |
| `usage(routing)` | a function: share of test tokens whose **first choice** is each expert, as a length-8 array |

### Output your code must print

```
MODEL           test acc   router moved   expert usage (share of test tokens)
dense FFN        0.____         -          -
noisy top-2      0.____       _.___        [0.__ 0.__ 0.__ 0.__ 0.__ 0.__ 0.__ 0.__]
noisy top-1      0.____       _.___        [ ... ]
switch top-1     0.____       _.___        [ ... ]
```

### Then answer

**Q1.** One of the three routers did not move **at all**. Which one, and why? Use the gradient from the
lecture's backpropagation slide, $\nabla L / \nabla G(x)_i = E_i(x)^T\, \nabla L / \nabla y$, and work out
what $G(x)$ is for the chosen expert when $k = 1$. Why does the Switch router not have this problem?
Finally: that model's accuracy is not much worse than the others — what does that tell you about
this task?

### Hints

- `W0 = model.ffn.router.w_gate.weight.detach().clone()` **before** `train_model(...)`.
- `torch.cat(routing)[:, 0]` gives every test token's first-choice expert;
  `np.bincount(..., minlength=8)` counts each expert.
- In `SwitchRouter`, `top_idx` must still have shape `(T, 1)` so that `MoELayer` works unchanged.

In [3]:
# ---------------- TASK 2 ----------------

class SwitchRouter(nn.Module):
    def __init__(self, d_model, n_experts):
        super().__init__()
        self.w_gate = nn.Linear(d_model, n_experts, bias=False)
        self.k, self.n_experts = 1, n_experts

    def forward(self, x):
        probs = F.softmax(self.w_gate(x), dim=-1)                       # softmax over ALL experts
        top_idx = probs.argmax(dim=-1, keepdim=True)                    # (T, 1)
        gates = torch.zeros_like(probs).scatter(-1, top_idx, probs.gather(-1, top_idx))
        return gates, top_idx, probs


def usage(routing):
    first = torch.cat(routing)[:, 0].numpy()
    return np.bincount(first, minlength=N_EXPERTS) / len(first)


def train_and_measure(make_router, alpha=0.0):
    model = make_model(lambda: MoELayer(D_MODEL, D_HIDDEN, N_EXPERTS, make_router()), seed=42)
    W0 = model.ffn.router.w_gate.weight.detach().clone()               # BEFORE training
    train_model(model, alpha=alpha)
    moved = (model.ffn.router.w_gate.weight.detach() - W0).norm().item()
    acc, routing = evaluate(model)
    return model, moved, acc, routing

results = {}
print("noisy top-2:");  noisy_k2,  *results['noisy top-2']  = train_and_measure(lambda: NoisyTopKRouter(D_MODEL, N_EXPERTS, k=2))
print("noisy top-1:");  noisy_k1,  *results['noisy top-1']  = train_and_measure(lambda: NoisyTopKRouter(D_MODEL, N_EXPERTS, k=1))
print("switch top-1:"); switch_a0, *results['switch top-1'] = train_and_measure(lambda: SwitchRouter(D_MODEL, N_EXPERTS))

fmt = lambda u: '[' + ' '.join(f'{v:.2f}' for v in u) + ']'
print()
print("MODEL           test acc   router moved   expert usage (share of test tokens)")
print(f"{'dense FFN':<15}  {dense_acc:.4f}         -          -")
for name, (moved, acc, routing) in results.items():
    print(f"{name:<15}  {acc:.4f}       {moved:.3f}        {fmt(usage(routing))}")


noisy top-2:


  trained in 8 s
noisy top-1:


  trained in 7 s
switch top-1:


  trained in 7 s

MODEL           test acc   router moved   expert usage (share of test tokens)
dense FFN        0.8505         -          -
noisy top-2      0.8435       1.476        [0.05 0.09 0.12 0.11 0.26 0.10 0.19 0.08]
noisy top-1      0.8480       0.000        [0.10 0.16 0.09 0.08 0.14 0.17 0.10 0.16]
switch top-1     0.8385       1.512        [0.14 0.11 0.12 0.15 0.13 0.11 0.13 0.10]


---
# Task 3: The load-balancing loss &nbsp;&nbsp;*(4 marks)*

The tutorial's section 5 ended with a router that overloaded one expert. Look at your Task 2 table:
the **noisy top-2** router is the uneven one — compare its busiest and quietest experts with an even
share of $1/8 = 0.125$. The Switch Transformer's fix is an extra loss term (the *Load Balancing Loss*
slide in part 2). For a batch with $T$ tokens and $N$ experts:

$$f_i = \frac{1}{T}\sum_{x \in \text{batch}} \mathbb{1}\{\arg\max p(x) = i\}
\qquad\quad P_i = \frac{1}{T}\sum_{x \in \text{batch}} p_i(x)
\qquad\quad \text{loss} = \alpha \cdot N \cdot \sum_{i=1}^{N} f_i\,P_i$$

$f_i$ is the **fraction of tokens dispatched** to expert $i$ (with top-2 routing, count each token's
**first** choice — that is `self.top1`); $P_i$ is the **average router probability** for expert $i$.
When every expert gets exactly $1/N$ of the tokens, the sum is exactly 1. `train_model` already adds
`alpha * load_balancing_loss(...)` when `alpha > 0` — you write the function.

### What your code must create

| Name | Is |
|---|---|
| `load_balancing_loss(probs, top1, n_experts)` | returns the scalar $N \sum_i f_i P_i$ (without $\alpha$) |
| `noisy_k2_a001`, `noisy_k2_a01` | **noisy top-2** models trained with `alpha=0.01` and `alpha=0.1` (`noisy_k2` from Task 2 is the `alpha=0` one) |

### Output your code must print

First two sanity checks on made-up routings with 8 experts:

```
perfectly balanced: 1.000
everything to expert 0 (probability 1): 8.000
```

Then the comparison, where **usage CV** is the coefficient of variation of the usage array
(standard deviation ÷ mean: 0 means perfectly even):

```
alpha    test acc   usage CV   busiest expert   quietest expert
0.00      0.____     0.___        0.___           0.___
0.01      0.____     0.___        0.___           0.___
0.10      0.____     0.___        0.___           0.___
```

### Then answer

**Q2.** Did the loss even out the routing, and what did it cost in accuracy? $f_i$ is computed with an
argmax, so no gradient flows through it — so which part of the loss actually trains the router, and
why does multiplying by $f_i$ still push tokens **away** from busy experts?

### Hints

- `F.one_hot(top1, n_experts).float().mean(0)` gives $f$; `probs.mean(0)` gives $P$.
- For the balanced check: 8 tokens, token $t$ sent to expert $t$, and `probs` all equal to $1/8$.
- Train with `train_model(model, alpha=...)`, building each model with `make_model(..., seed=42)`.

In [4]:
# ---------------- TASK 3 ----------------

def load_balancing_loss(probs, top1, n_experts):
    f = F.one_hot(top1, n_experts).float().mean(0)      # fraction of tokens dispatched (no gradient)
    P = probs.mean(0)                                    # mean router probability (differentiable)
    return n_experts * (f * P).sum()


# sanity checks
print(f"perfectly balanced: {load_balancing_loss(torch.full((8, 8), 1/8), torch.arange(8), 8).item():.3f}")
all0 = torch.zeros(8, 8); all0[:, 0] = 1.0
print(f"everything to expert 0 (probability 1): {load_balancing_loss(all0, torch.zeros(8, dtype=torch.long), 8).item():.3f}")
print()

print("noisy top-2, alpha = 0.01:")
noisy_k2_a001, _, acc_a001, routing_a001 = train_and_measure(lambda: NoisyTopKRouter(D_MODEL, N_EXPERTS, k=2), alpha=0.01)
print("noisy top-2, alpha = 0.1:")
noisy_k2_a01,  _, acc_a01,  routing_a01  = train_and_measure(lambda: NoisyTopKRouter(D_MODEL, N_EXPERTS, k=2), alpha=0.1)
_, acc_a0, routing_a0 = results['noisy top-2']

print()
print("alpha    test acc   usage CV   busiest expert   quietest expert")
for a, acc, routing in [(0.0, acc_a0, routing_a0), (0.01, acc_a001, routing_a001), (0.1, acc_a01, routing_a01)]:
    u = usage(routing)
    print(f"{a:.2f}      {acc:.4f}     {u.std() / u.mean():.3f}        {u.max():.3f}           {u.min():.3f}")


perfectly balanced: 1.000
everything to expert 0 (probability 1): 8.000

noisy top-2, alpha = 0.01:


  trained in 8 s
noisy top-2, alpha = 0.1:


  trained in 8 s

alpha    test acc   usage CV   busiest expert   quietest expert
0.00      0.8435     0.503        0.261           0.053
0.01      0.8420     0.219        0.182           0.102
0.10      0.8435     0.130        0.161           0.105


---
# Task 4: Expert capacity and dropped tokens &nbsp;&nbsp;*(4 marks)*

Why does balance matter if accuracy barely moves? Because real MoE models put each expert on its own
device, and every device needs a **fixed-size** buffer (the *How to set Expert Capacity?* slides in
part 2). For a batch of $T$ tokens:

$$\text{capacity} = \left\lfloor \text{capacity factor} \times \frac{T \cdot k}{N} \right\rfloor$$

($T \cdot k$ because with top-$k$ routing every token takes up $k$ expert slots.) If more tokens than
that are routed to an expert, the extra ones **overflow**: they are **dropped** from that expert, skip
it entirely, and continue only through the residual connection. A capacity factor of 1.0 leaves no
slack at all; Switch Transformer used 1.25.

You do not need to change the model. `evaluate` already returns the routing of each test batch; count
how many tokens would overflow.

### What your code must create

| Name | Is |
|---|---|
| `dropped_fraction(routing, n_experts, capacity_factor)` | fraction of all token-to-expert assignments that overflow, computed **batch by batch** |

### Output your code must print

A check you can do by hand (the slides' example: 6 tokens, 3 experts, capacity factor 1.0):

```
toy check: 0.167
```

Then, for the noisy top-2 models trained with `alpha = 0` and `alpha = 0.1`:

```
capacity factor   dropped (alpha=0.00)   dropped (alpha=0.10)
1.00                    _._%                   _._%
1.25                    _._%                   _._%
1.50                    _._%                   _._%
2.00                    _._%                   _._%
```

### Then answer

**Q3.** The balanced model still drops tokens at capacity factor 1.0 — why, if its overall usage is
nearly even? What happens to a dropped token, and why is that not a disaster for accuracy here? Why not
just set the capacity factor to 2.0?

### Hints

- For the toy check use `routing = [torch.tensor([[0], [0], [0], [1], [2], [2]])]` — 6 tokens, $k = 1$,
  3 experts: capacity is $\lfloor 1.0 \times 6 \cdot 1/3 \rfloor = 2$, expert 0 receives 3, so one
  assignment in six overflows.
- Each routing tensor has shape `(T, k)`. `torch.bincount(r.flatten(), minlength=n_experts)` counts
  every assignment per expert; `(counts - capacity).clamp(min=0).sum()` counts the overflow.

In [5]:
# ---------------- TASK 4 ----------------

def dropped_fraction(routing, n_experts, capacity_factor):
    dropped, total = 0, 0
    for r in routing:                                    # capacity is a PER-BATCH limit
        T, k = r.shape
        capacity = int(capacity_factor * T * k / n_experts)
        counts = torch.bincount(r.flatten(), minlength=n_experts)
        dropped += (counts - capacity).clamp(min=0).sum().item()
        total   += r.numel()
    return dropped / total


toy = [torch.tensor([[0], [0], [0], [1], [2], [2]])]
print(f"toy check: {dropped_fraction(toy, 3, 1.0):.3f}")
print()
print("capacity factor   dropped (alpha=0.00)   dropped (alpha=0.10)")
for cf in [1.0, 1.25, 1.5, 2.0]:
    d0  = dropped_fraction(routing_a0,  N_EXPERTS, cf)
    d01 = dropped_fraction(routing_a01, N_EXPERTS, cf)
    print(f"{cf:.2f}                    {100*d0:4.1f}%                  {100*d01:4.1f}%")


toy check: 0.167

capacity factor   dropped (alpha=0.00)   dropped (alpha=0.10)
1.00                    10.4%                   5.8%
1.25                     3.5%                   0.2%
1.50                     0.6%                   0.0%
2.00                     0.0%                   0.0%


---
# Task 5: What did the experts learn? &nbsp;&nbsp;*(4 marks)*

The Mixtral slides in part 2 ask whether experts specialise — does one expert handle the maths, another
the code? Their answer for Mixtral was **no**: experts did not specialise by domain, but consecutive
tokens were often sent to the same expert. Test both on your own model.

Use the **`alpha = 0.1` noisy top-2 model** (`noisy_k2_a01`) and each token's **first-choice**
expert.

### What your code must create and print

**(a) Topic × expert table.** For each topic, the share of **that topic's tokens** sent to each expert
(each row adds up to 1). Then each topic's favourite expert and its share — compare with an even
spread, $1/8 = 0.125$.

```
            E0    E1    E2    E3    E4    E5    E6    E7
World      0.__  0.__  ...
Sports     ...
Business   ...
Sci/Tech   ...

favourite expert:  World E_ (0.__)   Sports E_ (0.__)   Business E_ (0.__)   Sci/Tech E_ (0.__)
```

**(b) Top tokens per expert.** The 10 most frequent tokens routed to each expert.

```
E0 (0.__ of tokens): ['.', 'the', ...]
...
```

**(c) Consecutive tokens.** Of all pairs of neighbouring tokens **within the same article**, the
fraction routed to the same expert — and what chance would give, $\sum_i u_i^2$ where $u$ is the usage
array.

```
consecutive tokens on the same expert: 0.___   (chance: 0.___)
```

### Then answer

**Q4.** Do your experts specialise by topic? Use the table: is any expert a "Sports expert"? What do the
top-token lists suggest the router actually groups by? Compare with the Mixtral slide.

**Q5.** Consecutive tokens share an expert more often than chance. Why might that be, given what the
self-attention layer **before** the MoE layer does to each token's vector?

### Hints

- Take each routing tensor's first column, `r[:, 0]`. The rows follow `X_batch[X_batch != 0]` exactly — the same row-by-row order the model
  uses — so that expression gives the token ids that line up with each routing tensor.
- `torch.repeat_interleave(y_batch, (X_batch != 0).sum(1))` gives each token its article's topic.
- Padding is at the **end** of every article, so each article's tokens are one consecutive run in the
  routing tensor; `(X_batch != 0).sum(1)` gives the run lengths.

In [6]:
# ---------------- TASK 5 ----------------

acc5, routing5 = evaluate(noisy_k2_a01)
tok_ids, tok_topic, tok_expert, run_lengths = [], [], [], []
for bi, b in enumerate(range(0, len(X_test), BATCH)):
    Xb, yb = X_test[b:b + BATCH], y_test[b:b + BATCH]
    m = Xb != 0
    tok_ids.append(Xb[m])                                            # same row-by-row order as the model
    tok_topic.append(torch.repeat_interleave(yb, m.sum(1)))
    tok_expert.append(routing5[bi][:, 0])                            # first choice
    run_lengths.append(m.sum(1))
tok_ids, tok_topic, tok_expert, run_lengths = map(torch.cat, (tok_ids, tok_topic, tok_expert, run_lengths))
assert len(tok_ids) == len(tok_expert)

# (a) topic x expert table
table = np.zeros((len(TOPICS), N_EXPERTS))
for t in range(len(TOPICS)):
    e = tok_expert[tok_topic == t].numpy()
    table[t] = np.bincount(e, minlength=N_EXPERTS) / len(e)
print("(a) share of each topic's tokens sent to each expert\n")
print("          " + "".join(f"  E{i}  " for i in range(N_EXPERTS)))
for t, name in enumerate(TOPICS):
    print(f"{name:<9} " + "".join(f" {v:.2f} " for v in table[t]))
print("\nfavourite expert:  " + "   ".join(
    f"{name} E{table[t].argmax()} ({table[t].max():.2f})" for t, name in enumerate(TOPICS)))

# (b) top tokens per expert
print("\n(b) the 10 most frequent tokens routed to each expert\n")
u = np.bincount(tok_expert.numpy(), minlength=N_EXPERTS) / len(tok_expert)
for i in range(N_EXPERTS):
    c = collections.Counter(tok_ids[tok_expert == i].tolist())
    print(f"E{i} ({u[i]:.2f} of tokens): {[vocab[w] for w, _ in c.most_common(10)]}")

# (c) consecutive tokens within the same article
same, pairs, start = 0, 0, 0
for L in run_lengths.tolist():
    e = tok_expert[start:start + L]
    same  += (e[1:] == e[:-1]).sum().item()
    pairs += L - 1
    start += L
print(f"\n(c) consecutive tokens on the same expert: {same / pairs:.3f}   (chance: {(u ** 2).sum():.3f})")


(a) share of each topic's tokens sent to each expert

            E0    E1    E2    E3    E4    E5    E6    E7  
World      0.12  0.09  0.09  0.24  0.12  0.06  0.09  0.20 
Sports     0.24  0.09  0.03  0.07  0.29  0.06  0.04  0.18 
Business   0.07  0.16  0.25  0.08  0.09  0.16  0.12  0.07 
Sci/Tech   0.12  0.14  0.05  0.04  0.15  0.24  0.24  0.04 

favourite expert:  World E3 (0.24)   Sports E4 (0.29)   Business E2 (0.25)   Sci/Tech E5 (0.24)

(b) the 10 most frequent tokens routed to each expert

E0 (0.14 of tokens): ['the', 'to', ',', '.', 'for', 'a', '(', 'his', 'and', 'game']
E1 (0.12 of tokens): ['.', '-', 'ap', 'by', 'and', 'in', '(', 'it', 'at', 'be']
E2 (0.10 of tokens): ['a', 'for', 'the', ',', 'to', '/', '-', 'its', 'us', 'oil']
E3 (0.11 of tokens): [',', '<unk>', 'of', 's', 'and', 'said', 'on', 'iraq', 'over', '.']
E4 (0.16 of tokens): ['<unk>', '-', 'to', ',', "'", 'and', '.', 'the', 'a', 'has']
E5 (0.13 of tokens): ['.', ',', 'in', 'as', 'to', 'new', '<unk>', 'a', 'the', 'o

---
## Your answers

**Q1.** *(Task 2)* The **noisy top-1** router did not move at all (router moved = **0.000**), while noisy top-2 moved 1.476 and the Switch router moved 1.512.

The router only gets a gradient through the gate values, $\nabla L / \nabla G(x)_i = E_i(x)^T\,\nabla L / \nabla y$, and then through $\partial G / \partial H$. With $k = 1$ the KeepTopK vector has a single finite entry and every other entry is $-\infty$, so $G(x) = \text{Softmax}(\ldots) = e^{v}/e^{v} = 1$ for the chosen expert **whatever the logit $v$ is**. $G$ is a constant, so $\partial G/\partial H = 0$, and no gradient reaches $W_g$ (or $W_{noise}$). The argmax that picks the expert is not differentiable either. The router stays exactly at its random initialisation. The Switch router avoids this because its gate is the softmax probability over **all** experts, $p(x)_{i^*} = e^{h_{i^*}}/\sum_j e^{h_j}$. That value depends on every logit, so the loss can raise or lower $p_{i^*}$, and $W_g$ learns (1.512).

The frozen router still reached **0.8480**, about the same as noisy top-2 (0.8435), Switch (0.8385) and even the dense FFN (0.8505). A fixed random linear router is a consistent hash of the token vectors, and the experts simply learn to handle whichever tokens they are given. So on this task the routing decisions barely matter. Topic classification from ~44 tokens is easy, and most of the work is done by the embeddings, attention and mean-pooling. A learned router has little to add here.

**Q2.** *(Task 3)* Yes. The usage CV fell from **0.503** (α = 0) to **0.219** (α = 0.01) and **0.130** (α = 0.1). The busiest expert dropped from 0.261 to 0.161, and the quietest rose from 0.053 to 0.105, close to the even 0.125. It cost nothing in accuracy: 0.8435 → 0.8420 → 0.8435, which is within noise.

$f_i$ comes from an argmax and one-hot, so it has no gradient and acts as a **constant weight**. Only $P_i = \frac1T\sum_x p_i(x)$ trains the router: $\partial\,\text{loss}/\partial p_i(x) = \alpha N f_i / T$. Every token is pushed to lower its probability for expert $i$ in proportion to how busy expert $i$ already is ($f_i$). Busy experts get the strongest push down and quiet ones (small $f_i$) almost none. Because the softmax sums to 1, the probability taken from busy experts moves to quiet ones. As the logits shift, some tokens' argmax flips to a quieter expert, which lowers the next batch's $f$ for the busy expert. The non-differentiable $f$ decides **where** to push, and the differentiable $P$ does the pushing.

**Q3.** *(Task 4)* At capacity factor 1.0 the balanced model (α = 0.1) still drops **5.8%** of assignments, compared with 10.4% for α = 0. It drops 0.2% at 1.25 and nothing from 1.5. Capacity 1.0 leaves no slack: an expert overflows as soon as it gets even one token more than exactly $T k / N$ **in that batch**. "Nearly even" usage is an average over the whole test set. The busiest expert still gets 0.161 of first choices, about 1.3 × the fair share. Each batch of 64 articles also has its own topic mix (and Task 5 shows topics lean towards certain experts), so the per-batch load varies around the average and some expert is over its limit in most batches.

A dropped token **skips that expert**. Its contribution to $y_t$ is lost, and the token passes on through the residual connection ($x + 0$) and, with $k = 2$, possibly through its other expert. That is not a disaster here. The model is a single block and the classifier mean-pools ~44 tokens, so a few tokens losing their FFN update barely changes the pooled vector, and attention has already mixed in context. A capacity factor of 2.0 removes all drops, but every expert then reserves a buffer twice the fair share. That doubles the memory, the (mostly empty, padded) expert compute and the all-to-all communication between devices. You pay for 2 × the slots to save a few percent of tokens. A good balance loss plus a modest factor such as 1.25 gets nearly the same result (0.2% dropped) much more cheaply.

**Q4.** *(Task 5)* No expert specialises by topic in the sense of being a "Sports expert". There is a **mild lean**: each topic's favourite expert gets 0.24–0.29 of its tokens (World E3 0.24, Sports E4 0.29, Business E2 0.25, Sci/Tech E5 0.24), about twice the even 0.125. But that favourite is never close to owning the topic: 70–76% of each topic's tokens go elsewhere. The same experts also serve other topics heavily. E4 takes 0.29 of Sports but also 0.15 of Sci/Tech and 0.12 of World, and E0 is shared by Sports (0.24), World (0.12) and Sci/Tech (0.12). Every expert takes at least some tokens from every topic.

The top-token lists show what the router really groups by. Every expert's most frequent tokens are function words and punctuation (`the`, `to`, `,`, `.`, `of`, `a`, `-`, `<unk>`), shared across all experts. Only a few topical words appear (`game` in E0, `oil`/`us` in E2, `iraq`/`said` in E3, `reuters`/`ap` source tags in E6/E1). The router groups by **token identity and local, syntactic context**, not by domain. The mild topic lean comes from attention mixing some article context into each token. This matches the Mixtral slide: experts do not specialise by domain, and routing follows surface and syntactic features of tokens.

**Q5.** *(Task 5)* Consecutive tokens share an expert **0.186** of the time, compared with **0.127** by chance, about 1.5 × chance. Routing happens **after** self-attention, and attention replaces each token's vector with $x_t + \sum_s a_{ts} v_s$, a weighted mix of the other tokens in the article. Neighbouring tokens attend to largely the same context (the same article, similar positions, the same nearby words), so their post-attention vectors share a large common component. The positional embeddings of adjacent positions are also similar. Since the router is just a linear map $xW_g$ followed by an argmax, vectors that point in similar directions land on the same expert more often, so neighbours share an expert more often than chance. This is the locality effect the Mixtral analysis reported.


---
## Submission checklist

- **Runtime → Restart and run all** finishes without errors, with `mini_agnews.tsv` beside the notebook.
- Every task prints the output shown in its *Output your code must print* box.
- Q1–Q5 are answered in the cell above, in your own words, using **your** numbers.
- Submit the `.ipynb` with all outputs saved.

### Common mistakes

- **Adding noise at evaluation time.** The noise is for training only — check `self.training`.
- **Running every expert on every token** and multiplying by zero. It gives the right numbers but it
  is not a sparse layer, and Task 1 asks for sparse dispatch.
- **`gates` rows that do not sum to 1** — usually a softmax over all logits instead of over the
  KeepTopK vector.
- **Forgetting `self.probs` / `self.top_idx` / `self.top1`** in `MoELayer.forward`; Tasks 2–5 read them.
- **Copying the router weights after training** (or without `.clone()`), which makes every router
  look as if it never moved.
- **Computing dropped tokens over the whole test set at once** instead of batch by batch — capacity is
  a per-batch limit.
- **Pairing tokens across two different articles** in Task 5(c).